In [3]:
import numpy as np, os
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score, roc_curve,
                             mean_squared_error)

def evaluate_binary(y_true, y_pred, y_proba=None):
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2,2):
        tn, fp, fn, tp = cm.ravel()
        fpr = fp / (fp + tn) if (fp + tn) else 0.0
        tpr = tp / (tp + fn) if (tp + fn) else 0.0
    else:
        fpr = tpr = 0.0

    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall (TPR)": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "FPR": fpr,
        "TPR": tpr,
        "MSE": mean_squared_error(y_true, y_pred),
    }
    if y_proba is not None:
        try:
            metrics["AUC-ROC"] = roc_auc_score(y_true, y_proba)
        except Exception:
            metrics["AUC-ROC"] = float("nan")
    return metrics, cm

def plot_roc(y_true, y_proba, title="ROC"):
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    plt.figure()
    plt.plot(fpr, tpr, label="ROC")
    plt.plot([0,1],[0,1],'--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend()
    plt.show()


In [2]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200, n_jobs=-1, random_state=42, class_weight="balanced"
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:,1]

metrics_rf, cm_rf = evaluate_binary(y_test, y_pred, y_proba)
print("Confusion matrix (RF):\n", cm_rf)
for k,v in metrics_rf.items():
    print(f"{k:12s}: {v:.4f}")
plot_roc(y_test, y_proba, title="ROC — RandomForest")



NameError: name 'X_train' is not defined

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=2000, class_weight="balanced")
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:,1]

metrics_lr, cm_lr = evaluate_binary(y_test, y_pred_lr, y_proba_lr)
print("Confusion matrix (LR):\n", cm_lr)
for k,v in metrics_lr.items():
    print(f"{k:12s}: {v:.4f}")
plot_roc(y_test, y_proba_lr, title="ROC — LogisticRegression")


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

n_features = X_train.shape[1]
ann = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])
ann.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

es = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor="val_loss")
hist = ann.fit(X_train, y_train, epochs=10, batch_size=1024, validation_split=0.1, callbacks=[es], verbose=2)

y_proba_dl = ann.predict(X_test, verbose=0).ravel()
y_pred_dl = (y_proba_dl >= 0.5).astype(int)

metrics_dl, cm_dl = evaluate_binary(y_test, y_pred_dl, y_proba_dl)
print("Confusion matrix (Keras ANN):\n", cm_dl)
for k,v in metrics_dl.items():
    print(f"{k:12s}: {v:.4f}")
plot_roc(y_test, y_proba_dl, title="ROC — Keras ANN")


In [ ]:
import pandas as pd

rows = [
    {"Model":"RandomForest", **metrics_rf},
    {"Model":"LogisticRegression", **metrics_lr},
    {"Model":"KerasANN", **metrics_dl},
]
pd.DataFrame(rows).set_index("Model").round(4)
